# Querying Ecological Data with Kinship Earth

**Kinship Earth** is an ecological intelligence middleware — a unified API that makes ecological data from multiple authoritative sources queryable through a single interface. It combines:

- **OBIS** — 168M+ marine species occurrence records
- **NEON** — 81 US terrestrial monitoring sites with 180+ standardized data products
- **ERA5** — Global climate reanalysis data from 1940 to near-present (via Open-Meteo)
- **iNaturalist** — Community biodiversity observations across all taxa
- **eBird** — Real-time bird observation data

This notebook demonstrates how to connect to a Kinship Earth MCP server from Python and query ecological data programmatically. You will:

1. Connect to the MCP server
2. Search for species observations near Monterey Bay
3. Convert results to a pandas DataFrame for analysis
4. Visualize species distributions and climate data
5. Export results as GeoJSON for use in GIS tools

**Prerequisites:** Python 3.10+, a running Kinship Earth MCP server (local or remote).

## 1. Install Dependencies

The `mcp` package is the official Python client for the Model Context Protocol. We also use `httpx` for HTTP transport, `pandas` for data manipulation, and `matplotlib` for visualization.

In [ ]:
%pip install mcp httpx pandas matplotlib

## 2. Connect to the Kinship Earth MCP Server

There are two ways to connect:

- **Option A: stdio** — launch the server as a subprocess (best for local development)
- **Option B: HTTP** — connect to a running server over HTTP (best for shared/remote servers)

Uncomment the option that matches your setup.

In [ ]:
import json
from mcp import ClientSession

# --------------------------------------------------------------------------
# Option A: stdio transport (local server)
# The server is launched as a subprocess. Adjust the path to your installation.
# --------------------------------------------------------------------------
from mcp.client.stdio import StdioServerParameters, stdio_client

server_params = StdioServerParameters(
    command="uv",
    args=["run", "--directory", "/path/to/kinship-earth-mcp/servers/orchestrator",
          "python", "-m", "kinship_orchestrator.server"],
    env={},  # Add NEON_API_TOKEN or EBIRD_API_KEY here if needed
)

# We'll use this context manager pattern throughout the notebook.
# Each cell that calls a tool will open and close a session.
# In practice, you may want to keep a single long-lived session.

async def get_stdio_session():
    """Create an MCP client session over stdio."""
    transport = await stdio_client(server_params).__aenter__()
    read_stream, write_stream = transport
    session = ClientSession(read_stream, write_stream)
    await session.initialize()
    return session, transport

# --------------------------------------------------------------------------
# Option B: HTTP transport (remote server)
# Connect to a Kinship Earth server running in HTTP mode.
# --------------------------------------------------------------------------
# from mcp.client.streamable_http import streamablehttp_client
#
# KINSHIP_URL = "http://localhost:8000/mcp"  # Or your Railway deployment URL
#
# async def get_http_session():
#     """Create an MCP client session over HTTP."""
#     transport = await streamablehttp_client(KINSHIP_URL).__aenter__()
#     read_stream, write_stream = transport
#     session = ClientSession(read_stream, write_stream)
#     await session.initialize()
#     return session, transport

print("Connection helpers defined. Ready to query.")

## 3. Search for Species near Monterey Bay

The `ecology_search` tool is the main entry point. It queries OBIS (marine species), NEON (terrestrial monitoring sites), and ERA5 (climate) in a single call.

We'll search for all species observations within 50 km of Monterey Bay, California.

In [ ]:
from mcp.client.stdio import stdio_client

# Search parameters
MONTEREY_BAY = {"lat": 36.6, "lon": -121.9, "radius_km": 50}

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # List available tools (optional — useful for exploration)
        tools = await session.list_tools()
        print(f"Available tools: {[t.name for t in tools.tools]}")

        # Call ecology_search
        result = await session.call_tool(
            "ecology_search",
            arguments={
                "lat": MONTEREY_BAY["lat"],
                "lon": MONTEREY_BAY["lon"],
                "radius_km": MONTEREY_BAY["radius_km"],
                "limit": 50,
                "include_climate": True,
            },
        )

        # The result content is a list of TextContent objects
        search_data = json.loads(result.content[0].text)

print(f"Keys in response: {list(search_data.keys())}")
print(f"Species occurrences returned: {len(search_data.get('species_occurrences', []))}")
print(f"NEON sites found: {len(search_data.get('neon_sites', []))}")

## 4. Convert Results to a pandas DataFrame

The `species_occurrences` field contains a list of observation records. Each record includes taxonomic information, coordinates, date, and data quality metadata. Let's convert this to a DataFrame for easier analysis.

In [ ]:
import pandas as pd

# Convert species occurrences to DataFrame
occurrences = search_data.get("species_occurrences", [])
df = pd.DataFrame(occurrences)

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 records:")
df.head()

## 5. Visualize Species Observations

Let's create two views:
1. **Species count by taxonomic group** — which taxa are most represented?
2. **Observation locations on a scatter plot** — spatial distribution of observations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Chart 1: Species count by higher-level taxon ---
# The 'phylum' or 'class' field gives us a coarse taxonomic grouping.
# Adjust the column name based on what your server returns.
taxon_col = "phylum" if "phylum" in df.columns else "class" if "class" in df.columns else "scientific_name"
taxon_counts = df[taxon_col].value_counts().head(10)

taxon_counts.plot.barh(ax=axes[0], color="#2E86AB")
axes[0].set_xlabel("Number of Observations")
axes[0].set_ylabel(taxon_col.replace("_", " ").title())
axes[0].set_title("Species Observations by Taxon")
axes[0].invert_yaxis()  # Highest count at top

# --- Chart 2: Observation locations ---
lat_col = "lat" if "lat" in df.columns else "decimalLatitude"
lon_col = "lon" if "lon" in df.columns else "lng" if "lng" in df.columns else "decimalLongitude"

if lat_col in df.columns and lon_col in df.columns:
    axes[1].scatter(df[lon_col], df[lat_col], alpha=0.6, s=20, c="#A23B72")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    axes[1].set_title(f"Observation Locations — Monterey Bay ({len(df)} records)")

    # Mark the search center
    axes[1].scatter(
        MONTEREY_BAY["lon"], MONTEREY_BAY["lat"],
        marker="*", s=200, c="gold", edgecolors="black", zorder=5, label="Search center"
    )
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No coordinate columns found", ha="center", va="center")

plt.tight_layout()
plt.show()

## 6. Get Environmental Context

The `ecology_get_environmental_context` tool returns climate data and nearby monitoring sites for a specific location and date. This is useful for understanding the environmental conditions at the time of an observation.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        env_result = await session.call_tool(
            "ecology_get_environmental_context",
            arguments={
                "lat": 36.6,
                "lon": -121.9,
                "date": "2025-06-15",
                "days_before": 14,  # Two weeks of climate context
                "days_after": 0,
            },
        )

        env_data = json.loads(env_result.content[0].text)

print(f"Response keys: {list(env_data.keys())}")

# Show climate summary
if "climate" in env_data:
    climate = env_data["climate"]
    print(f"\nClimate data period: {climate.get('period', 'N/A')}")
    if "daily" in climate:
        print(f"Days of climate data: {len(climate['daily'])}")

## 7. Plot Climate Data

The climate response includes daily summaries with temperature, precipitation, wind, and solar radiation. Let's plot the temperature trend.

In [ ]:
# Extract daily climate data into a DataFrame
climate_daily = env_data.get("climate", {}).get("daily", [])
climate_df = pd.DataFrame(climate_daily)

if not climate_df.empty and "date" in climate_df.columns:
    climate_df["date"] = pd.to_datetime(climate_df["date"])

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # --- Temperature ---
    temp_cols = [c for c in climate_df.columns if "temperature" in c.lower() or "temp" in c.lower()]
    if temp_cols:
        for col in temp_cols:
            label = col.replace("_", " ").title()
            axes[0].plot(climate_df["date"], climate_df[col], marker="o", markersize=3, label=label)
        axes[0].set_ylabel("Temperature (C)")
        axes[0].set_title("Daily Temperature — Monterey Bay")
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)

    # --- Precipitation ---
    precip_cols = [c for c in climate_df.columns if "precip" in c.lower() or "rain" in c.lower()]
    if precip_cols:
        axes[1].bar(climate_df["date"], climate_df[precip_cols[0]], color="#4A90D9", alpha=0.7)
        axes[1].set_ylabel("Precipitation (mm)")
        axes[1].set_title("Daily Precipitation")
        axes[1].grid(True, alpha=0.3)

    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()
else:
    print("No daily climate data available. Check the response structure above.")

## 8. GeoJSON Output for GIS Tools

Kinship Earth supports a `geojson` output format, which returns species occurrences as a [GeoJSON FeatureCollection](https://datatracker.ietf.org/doc/html/rfc7946). This is directly importable into QGIS, ArcGIS, Leaflet, Kepler.gl, and other mapping tools.

The GeoJSON output uses the standard WGS84 coordinate reference system (EPSG:4326).

In [ ]:
from pathlib import Path

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Request GeoJSON output format
        geojson_result = await session.call_tool(
            "ecology_search",
            arguments={
                "lat": 36.6,
                "lon": -121.9,
                "radius_km": 50,
                "limit": 50,
                "output_format": "geojson",
            },
        )

        geojson_data = json.loads(geojson_result.content[0].text)

# The GeoJSON FeatureCollection is under the 'species_occurrences_geojson' key
feature_collection = geojson_data.get("species_occurrences_geojson", {})
print(f"GeoJSON type: {feature_collection.get('type')}")
print(f"Number of features: {len(feature_collection.get('features', []))}")

# Preview one feature
if feature_collection.get("features"):
    print(f"\nExample feature:")
    print(json.dumps(feature_collection["features"][0], indent=2))

In [ ]:
# Save GeoJSON to file for import into QGIS, ArcGIS, or other GIS tools
output_path = Path("monterey_bay_observations.geojson")
output_path.write_text(json.dumps(feature_collection, indent=2))
print(f"GeoJSON saved to {output_path.resolve()}")
print(f"Open this file in QGIS: Layer > Add Layer > Add Vector Layer > select the .geojson file")

## Next Steps

Now that you can query Kinship Earth from a notebook, here are some ideas:

- **Multi-site comparison**: Query several locations and compare biodiversity metrics. See the `ecological-monitoring.ipynb` notebook for a worked example.
- **Time-series analysis**: Use `era5_get_daily_summary` for long-term climate trends at your study sites.
- **Species distribution modeling**: Combine occurrence data with environmental variables for SDM workflows.
- **QGIS integration**: Export GeoJSON files and overlay them with raster layers in QGIS.

For more information:
- [Kinship Earth on GitHub](https://github.com/christineyws-beep/kinship-earth-mcp)
- [MCP Python SDK documentation](https://github.com/modelcontextprotocol/python-sdk)
- [Model Context Protocol specification](https://modelcontextprotocol.io/)